In [1]:
import scipy 
import numpy as np

## 1. Single Coin
We have a coin that might be biased, we don't know the probability of heads

Define the result of a single coin flip as $X_i$ ~ $Bernoulli(p)$

How to proceed: 
1) Define a true parameter $p$ value (can be randomized without looking at it to keep it unknown)
2) Generate observed data (e.g. n = 200 coin flips)
3) Define the likelihood
4) Calculate the maximum likelihood estimator (MLE) for the probability $p$
5) Inference about the parameter $p$ via hypothesis testing:
    - Wald test
    - Likelihood ratio 

In [2]:
# 1) Define a true parameter $p$ value (can be randomized without looking at it to keep it unknown)
true_p = np.random.uniform(0, 1)

In [3]:
# 2) Generate observed data (e.g. n = 200 coin flips)
n = 200
X = np.random.choice([0, 1], size=n, replace=True, p=[1-true_p, true_p])

In [4]:
# 3) Define the likelihood

Since we have a single coin, we don't really care about the order of the sequence of flips, instead we care about the total amount

Define $X = \sum_{i}^n X_i$ ~ $Binomial(n, p)$

The likelihood is given by: $P(X | p) = \binom{n}{k} p^{k} (1-p)^{n-k}$ 

The log-likelihood is given by: $log P(X | p) = log \binom{n}{k}  + k \cdot log(p) + (n-k) \cdot log(1-p)$

The negative log-likelihood is given by: $-log P(X | p) = -log \binom{n}{k}  - k \cdot log(p) - (n-k) \cdot log(1-p)$

The MLE estiamtor can be calculated by maximizing the likelihood (or log-likelihood) or by minimizing the negative log-likelihood
$$
\begin{aligned}
\hat{\theta} &= \arg\min_{\theta} \left[ -\log P(X \mid \theta) \right] \\
             &= \arg\min_{\theta} \left[ -\log \left( \prod_{i=1}^{n} p^{x_i}(1-p)^{1-x_i} \right) \right] \\
             &= \arg\min_{\theta} \left[ -\sum_{i=1}^{n} \left( \log(p^{x_i}) + \log((1-p)^{1-x_i}) \right) \right] \\
             &= \arg\min_{\theta} \left[ -\sum_{i=1}^{n} \left( x_i \log p + (1-x_i)\log(1-p) \right) \right]
\end{aligned}
$$

This translates to taking the partial derivative of the negative log-likelihood with respect to the parameter $p$ and set it to 0:

$$
\begin{aligned}
0 &= \frac{\partial}{\partial p} \left[ -log P(X | p) \right] \\ 
  &= \frac{\partial}{\partial p} \left[ -log \binom{n}{k}  - k \cdot log(p) - (n-k) \cdot log(1-p) \right] \\ 
  &= \frac{\partial}{\partial p} \left[ - k \cdot log(p) - (n-k) \cdot log(1-p) \right] \\ 
  &= \frac{\partial}{\partial p} \left[ -\frac{k}{p} - \frac{(n-k)}{(1-p)} (-1) \right] \\ 
  &= \frac{\partial}{\partial p} \left[ -\frac{k}{p} + \frac{(n-k)}{(1-p)} \right] \\ 
\end{aligned}
$$
$$
\begin{aligned}
\frac{k}{p} &= \frac{(n-k)}{(1-p)} \\ 
\frac{k(1-p)}{p(1-p)} &= \frac{p(n-k)}{p(1-p)} \\ 
k(1-p) &= p(n-k) \\
k - kp &= pn - kp \\
k &= pn \\
\hat{p}_{\mathrm{MLE}} &= \frac{k}{n}
\end{aligned}
$$


In [5]:
# 4) Calculate the maximum likelihood estimator for the probability $p$
p_mle = np.mean(X)

# Similarly we can use the scipy.optimize.minimize method
def neg_log_likelihood(p, x):
    if p <= 0 or p >= 1: 
        return np.inf
    else: 
        return -np.sum(x * np.log(p) + (1-x) * np.log(1-p))

results = scipy.optimize.minimize(
    fun=neg_log_likelihood, 
    x0=np.array([0.5]),
    args=(X,),
    method='BFGS',
)

p_mle_scipy = results.x[0]
print(f"p_mle = {p_mle_scipy} : calculated p_mle = {p_mle}")

p_mle = 0.6299999911928388 : calculated p_mle = 0.63


/Users/giacomo/miniforge3/lib/python3.12/site-packages/scipy/optimize/_numdiff.py:686: RuntimeWarning: invalid value encountered in subtract
  df = [f_eval - f0 for f_eval in f_evals]


In [6]:
# 5) Inference about the parameter $p$ via hypothesis testing: Wald test

Variance for each $X_i$ is given by: $Var(X_i) = p(1-p)$

The estimator is found to be $\hat{p}_{\mathrm{MLE}} = k/n$

According to the Central Limit Theorem we have: $\hat{p}$ ~ $N \left( p, \frac{p(1-p)}{n} \right)$

Although the correct standard error on the estimator is given by $SE(\hat{p}) = \sqrt{\frac{p(1-p)}{n}}$

This being said, the true value for $p$ is unknown, hence we use the estimator to estimate SE: $\hat{SE}(\hat{p}) = \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$

In [7]:
print(f"Null hypothesis: p = 0.5")
print(f"Alternative hypothesis: p != 0.5")
print("Test type is defined by alternative hypothesis, in this case it is a double tail z-test")
print("By setting the statistical significance to 0.05 we have with each tail having area of 0.025")

alpha = 0.05
z_critical = scipy.stats.norm.ppf(1 - alpha / 2)
ste = np.sqrt(p_mle * (1 - p_mle) / n)
margin_of_error = (scipy.stats.norm.ppf(0.975)) * ste
ci_lower = p_mle - z_critical * ste
ci_upper = p_mle + z_critical * ste
print(f"z critical value: {z_critical:.6f}")
print(f"95% Wald CI: ({ci_lower:.6f}, {ci_upper:.6f})")

# Similary using z-score
p_null = 0.5
z_score = (p_mle - p_null) / ste
p_value = 2 * scipy.stats.norm.sf(abs(z_score))
print(f"z-score = {z_score}, p-value = {p_value}")
print("Reject H0: data in favor or biased coin") if p_value < alpha else print("Fail to reject H0: data not in favor of biased coin")

# It woudl be also possible to evaluate the p-value using the exact Binomial test

Null hypothesis: p = 0.5
Alternative hypothesis: p != 0.5
Test type is defined by alternative hypothesis, in this case it is a double tail z-test
By setting the statistical significance to 0.05 we have with each tail having area of 0.025
z critical value: 1.959964
95% Wald CI: (0.563088, 0.696912)
z-score = 3.8079147180858057, p-value = 0.00014014358406199545
Reject H0: data in favor or biased coin


In [8]:
# 5) Hypothesis testing: Likelihood ratio
LR = 2 * (-neg_log_likelihood(p_mle, X) - (-neg_log_likelihood(p_null, X)))
lr_p_value = scipy.stats.chi2.sf(LR, df=1)

print(f"LR statistic: {LR:.6f}")
print(f"LR p-value: {lr_p_value}")
print("Reject H0: data in favor or biased coin") if lr_p_value < alpha else print("Fail to reject H0: data not in favor of biased coin")

LR statistic: 13.676600
LR p-value: 0.00021714340953090972
Reject H0: data in favor or biased coin


In [9]:
true_p

0.620210328556067